In [0]:
spark.conf.set("spark.sql.shuffle.partitions", "auto")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

In [0]:
%pip install -q numpy==1.26.4 xgboost==3.0.0 scikit-learn==1.6.1 pandas==2.2.3 lz4==4.3.2 psutil==5.9.0 mlflow==3.0.1
dbutils.library.restartPython()

In [0]:
import pyspark.sql.functions as F
import joblib
import config

In [0]:
model_uri = config.model_uri

features = config.features

model_input = ["account_number",
    "theme_clean",
    "baskets_behavior__recency_rank",
    *features]

In [0]:
# Widen the hackathon output from the top 25 to the top 100 ranked themes per customer.
predict_input = (
    spark.table("ds_sandbox.next_uk_nextAds_predict_prod_ranked")
    .filter("simple_rules_rank <= 100")
    .select(*model_input)
    .withColumnRenamed("theme_clean", "theme")
    .repartition(int(spark.conf.get("spark.default.parallelism", "200")), "account_number")
)

print(f"Loading model: {model_uri}")

In [0]:
encoders = joblib.load("ranking_encoders.joblib")

prediction_schema = (
    predict_input.select(
        F.col("account_number"),
        F.col("theme"),
        F.col("month"),
        F.col("baskets_behavior__recency_rank"),
        F.lit(0.0).cast("float").alias("prediction"),
    )
    .schema
)

In [0]:
def predict_partition(iterator):
    import mlflow
    import numpy as np
    import xgboost as xgb

    mlflow.set_tracking_uri("databricks")

    partition_encoders = encoders

    global _hackathon_predict_model
    try:
        model = _hackathon_predict_model
    except NameError:
        _hackathon_predict_model = mlflow.xgboost.load_model(model_uri)
        model = _hackathon_predict_model

    for pdf in iterator:
        if pdf.empty:
            continue

        feature_pdf = pdf[features].copy()

        for col, encoder in partition_encoders.items():
            if col not in feature_pdf.columns:
                continue

            values = pdf[col].astype(str)
            valid_mask = values.isin(encoder.classes_)
            safe_values = np.where(valid_mask, values, encoder.classes_[0])
            encoded_values = encoder.transform(safe_values).astype(np.int32, copy=False)
            encoded_values[~valid_mask.to_numpy()] = -1
            feature_pdf[col] = encoded_values

        X = feature_pdf.to_numpy(dtype=np.float32, copy=False)
        dmatrix = xgb.QuantileDMatrix(X, feature_names=features)
        preds = model.predict(dmatrix).astype(np.float32, copy=False)

        result = pdf[["account_number", "theme", "month", "baskets_behavior__recency_rank"]].copy()
        result["prediction"] = preds
        yield result

In [0]:
predictions = predict_input.mapInPandas(predict_partition, schema=prediction_schema)

In [0]:
# display(predictions.limit(200))

In [0]:
output_table = "ds_sandbox.next_uk_nextAds_predict_prod_half"
# output_table

In [0]:
predictions.write.mode("overwrite").saveAsTable(output_table)

In [0]:
# Post-write validation — counts read from Delta metadata, not a full scan
input_count = (
    spark.table("ds_sandbox.next_uk_nextAds_predict_prod_ranked")
    .filter("simple_rules_rank <= 100")
    .count()
)
output_count = spark.table(output_table).count()

print(f"Input rows:  {input_count:,}")
print(f"Output rows: {output_count:,}")

if input_count != output_count:
    raise ValueError(f"Row count mismatch: {input_count:,} input vs {output_count:,} output")

print("Validation passed")

In [0]:
# display(spark.table(output_table).limit(20))